# Live Yahoo lineup workflow

This notebook uses the project modules directly: load config, fetch your Yahoo roster, enrich it with MLB starting-status checks, and then run the optimizer. By default it stays in dry-run mode; a later cell lets you opt into a live Yahoo write and refetch the roster.

In [1]:
from datetime import date
from pathlib import Path
import sys

repo_root = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import load_config
from lineup import (
    optimize_lineup,
    render_plan,
    render_roster,
    is_bench_position,
    player_should_be_replaced,
    choose_replacement,
    can_fill_position,
    apply_plan_to_roster,
    global_hitter_slot_value,
    pending_upgrade_value,
    normalized_average_pick_score,
    normalized_actual_rank_last_week_score,
    starting_tiebreak_score,
    slot_flexibility_bonus,
)
from mlb_lineups import clear_caches, enrich_roster_with_starting_status
from yahoo_api import YahooFantasyClient, build_roster_update_xml, build_matchup_delta_map
from projections import project_player_for_league_categories, category_urgency_weights, weighted_matchup_score, matchup_day_factor, build_hitter_matchup_adjustments


In [2]:
TARGET_DATE = '2026-04-01'
VERBOSE = False
IGNORE_LOCKS = False

config = load_config(lineup_date=TARGET_DATE, apply_changes=False)
client = YahooFantasyClient(config)
raw_roster = client.get_team_roster(config.lineup_date)
print(f'Fetched {len(raw_roster.players)} players for {raw_roster.team_name}.')


Fetched 29 players for deGrom Reapers.


In [3]:
clear_caches()
roster = enrich_roster_with_starting_status(raw_roster, date_str=config.lineup_date, verbose=VERBOSE, ignore_locks=IGNORE_LOCKS)
print(render_roster(roster))

Team: deGrom Reapers
Date: 2026-04-01
Slots: C x1, 1B x1, 2B x1, 3B x1, SS x1, IF x1, LF x1, CF x1, RF x1, OF x1, Util x1, SP x3, RP x3, P x3
Roster:
- C    Shea Langeliers (C) [lineup pending]
- 1B   Josh Naylor (1B) [lineup pending]
- 2B   Luis Arraez (1B,2B) [lineup pending]
- 3B   Alec Bohm (1B,3B) [lineup pending]
- SS   Zach Neto (SS) [lineup pending]
- IF   Elly De La Cruz (SS) [lineup pending]
- LF   Wyatt Langford (LF,CF) [lineup pending]
- CF   Pete Crow-Armstrong (CF) [lineup pending]
- RF   Jordan Walker (LF,RF) [lineup pending]
- OF   Owen Caissie (CF,RF) [lineup pending]
- Util Shohei Ohtani (Batter) (Util) [lineup pending]
- BN   Caleb Durbin (2B,3B) [lineup pending]
- BN   Colt Keith (1B,2B,3B) [lineup pending]
- BN   Dominic Canzone (LF,RF) [lineup pending]
- BN   Hunter Goodman (C) [lineup pending]
- IL   Jackson Chourio (LF,CF,RF) [inactive slot]
- NA   Bryce Eldridge (1B) [inactive slot]
- SP   Drew Rasmussen (SP) [starting]
- SP   Jacob deGrom (SP) [not starting]
-

In [4]:
matchup = client.get_current_matchup()
matchup_adjustments = {}
if matchup is not None:
    matchup_adjustments = build_hitter_matchup_adjustments(
        roster.players,
        date.fromisoformat(config.lineup_date),
        build_matchup_delta_map(matchup),
        verbose=VERBOSE,
    )

plan = optimize_lineup(roster, matchup_adjustments=matchup_adjustments)
print(render_plan(plan))

Plan:
- Warning: No eligible starting replacement found for Jacob deGrom at SP.
- No changes proposed.


In [5]:
DEBUG_PLAYERS = {"Luis Arraez", "Caleb Durbin"}
DEBUG_SLOT = "2B"

for p in roster.players:
    
    
    if p.name in DEBUG_PLAYERS:
        print(
            p.name,
            "| slot=", p.selected_position,
            "| start=", p.is_starting_today,
            "| reason=", p.starting_status_reason,
            "| %start=", p.yahoo_percent_started,
            "| avg_pick=", p.yahoo_average_pick,
            "| ar_lastweek=", p.yahoo_actual_rank_last_week,
            "| avg_pick_score=", normalized_average_pick_score(p),
            "| ar_lastweek_score=", normalized_actual_rank_last_week_score(p),
            "| tiebreak_score=", starting_tiebreak_score(p),
            "| flexibility_bonus=", slot_flexibility_bonus(p, DEBUG_SLOT),
            "| pending_value=", pending_upgrade_value(p, {}, DEBUG_SLOT),
            "| matchup_adjustment=", matchup_adjustments.get(p.player_key, 0),
            "| global_slot_score=", global_hitter_slot_value(p, {}, DEBUG_SLOT, matchup_adjustments),
        )


Luis Arraez | slot= 2B | start= None | reason= lineup_pending | %start= 39 | avg_pick= 197.2 | ar_lastweek= 3 | avg_pick_score= 61 | ar_lastweek_score= 90 | tiebreak_score= 190 | flexibility_bonus= 8 | pending_value= 198 | global_slot_score= 906
Caleb Durbin | slot= BN | start= None | reason= lineup_pending | %start= 12 | avg_pick= 196.7 | ar_lastweek= 13 | avg_pick_score= 61 | ar_lastweek_score= 40 | tiebreak_score= 113 | flexibility_bonus= 8 | pending_value= 113 | global_slot_score= 821


In [6]:
updated_roster = apply_plan_to_roster(roster, plan)
print(render_roster(updated_roster))


Team: deGrom Reapers
Date: 2026-04-01
Slots: C x1, 1B x1, 2B x1, 3B x1, SS x1, IF x1, LF x1, CF x1, RF x1, OF x1, Util x1, SP x3, RP x3, P x3
Roster:
- C    Shea Langeliers (C) [lineup pending]
- 1B   Josh Naylor (1B) [lineup pending]
- 2B   Luis Arraez (1B,2B) [lineup pending]
- 3B   Alec Bohm (1B,3B) [lineup pending]
- SS   Zach Neto (SS) [lineup pending]
- IF   Elly De La Cruz (SS) [lineup pending]
- LF   Wyatt Langford (LF,CF) [lineup pending]
- CF   Pete Crow-Armstrong (CF) [lineup pending]
- RF   Jordan Walker (LF,RF) [lineup pending]
- OF   Owen Caissie (CF,RF) [lineup pending]
- Util Shohei Ohtani (Batter) (Util) [lineup pending]
- BN   Caleb Durbin (2B,3B) [lineup pending]
- BN   Colt Keith (1B,2B,3B) [lineup pending]
- BN   Dominic Canzone (LF,RF) [lineup pending]
- BN   Hunter Goodman (C) [lineup pending]
- IL   Jackson Chourio (LF,CF,RF) [inactive slot]
- NA   Bryce Eldridge (1B) [inactive slot]
- SP   Drew Rasmussen (SP) [starting]
- SP   Jacob deGrom (SP) [not starting]
-

In [7]:
APPLY_TO_YAHOO = False

if not plan.moves:
    print('No changes to apply.')
else:
    print('Planned moves:')
    for move in plan.moves:
        print(f'- {move.player_name}: {move.from_position} -> {move.to_position}')
    print('\nOutgoing Yahoo roster XML:')
    print(build_roster_update_xml(roster.lineup_date or config.lineup_date, plan.moves))
    if APPLY_TO_YAHOO:
        client.set_lineup(
            lineup_date=roster.lineup_date or config.lineup_date,
            moves=plan.moves,
        )
        print('\nApplied lineup changes to Yahoo. Refetching roster...')
        refreshed_raw_roster = client.get_team_roster(config.lineup_date)
        clear_caches()
        refreshed_roster = enrich_roster_with_starting_status(
            refreshed_raw_roster,
            date_str=config.lineup_date,
            verbose=VERBOSE,
            ignore_locks=IGNORE_LOCKS,
        )
        print(render_roster(refreshed_roster))
    else:
        print('\nAPPLY_TO_YAHOO is False. No live changes sent.')


No changes to apply.


In [11]:
from datetime import date

PROJECTION_PLAYERS = {"Owen Caissie", "Luis Arraez"}

for p in roster.players:
    if p.name in PROJECTION_PLAYERS:
        projection = project_player_for_league_categories(p, date.fromisoformat(config.lineup_date), verbose=VERBOSE)
        print(p.name)
        print(projection)
        print()


Luis Arraez
PlayerProjection(player_key='469.p.11221', player_name='Luis Arraez', projection_type='hitting', source_window='70% last 30 days / 30% season per game', stats={'R': 0.0, 'H': 0.6, 'HR': 0.0, 'RBI': 0.2, 'SB': 0.2, 'K': 0.4, 'OPS': 0.367}, details={'last30_games': 5.0, 'season_games': 5.0, 'last30_ops': 0.367, 'season_ops': 0.367})

Owen Caissie
PlayerProjection(player_key='469.p.12127', player_name='Owen Caissie', projection_type='hitting', source_window='70% last 30 days / 30% season per game', stats={'R': 0.6, 'H': 1.2, 'HR': 0.2, 'RBI': 1.2, 'SB': 0.2, 'K': 0.8, 'OPS': 1.177}, details={'last30_games': 5.0, 'season_games': 5.0, 'last30_ops': 1.177, 'season_ops': 1.177})



In [9]:
from datetime import date

# Example weekly category gaps: positive means you are ahead, negative means you are behind.
CATEGORY_DELTAS = {
    "batting:R": 1,
    "batting:H": -1,
    "batting:HR": 2,
    "batting:RBI": 0,
    "batting:SB": 0,
    "batting:K": -2,
    "batting:OPS": -0.015,
}
urgency = category_urgency_weights(CATEGORY_DELTAS)
print("day_factor=", matchup_day_factor(date.fromisoformat(config.lineup_date)))
print("urgency=", urgency)
print()

for p in roster.players:
    if p.name in PROJECTION_PLAYERS:
        projection = project_player_for_league_categories(p, date.fromisoformat(config.lineup_date), verbose=VERBOSE)
        if projection is None:
            print(p.name, "-> no projection")
            continue
        print(p.name)
        print("stats=", projection.stats)
        print("matchup_score=", weighted_matchup_score(projection, urgency))
        print()


day_factor= 0.0
urgency= {'batting:R': 1.0, 'batting:H': 1.0, 'batting:HR': 0.5, 'batting:RBI': 1.0, 'batting:SB': 1.0, 'batting:K': 0.5, 'batting:OPS': 1.0}

Owen Caissie
stats= {'R': 0.6, 'H': 1.2, 'HR': 0.2, 'RBI': 1.2, 'SB': 0.2, 'K': 0.8, 'OPS': 1.177}
matchup_score= 4.077

Dominic Canzone
stats= {'R': 0.8, 'H': 0.6, 'HR': 0.4, 'RBI': 0.4, 'SB': 0.0, 'K': 0.2, 'OPS': 1.294}
matchup_score= 3.194



In [10]:
matchup = client.get_current_matchup()
if matchup is None:
    print("No current matchup found.")
else:
    print(f"Week {matchup.week}: {matchup.team_name} vs {matchup.opponent_team_name} ({matchup.status})")
    print(f"Points: {matchup.team_points} - {matchup.opponent_team_points}")
    print()
    delta_map = build_matchup_delta_map(matchup)
    for category_key, category in delta_map.items():
        print(
            f"stat_id={category.stat_id:>3} | {category_key:<14} | me={category.my_raw_value:<8} | opp={category.opponent_raw_value:<8} | delta={category.delta}"
        )


Week 2: deGrom Reapers vs Vladddys Daddys Daddy (midevent)
Points: 4.0 - 9.0

stat_id=  7 | batting:R      | me=7        | opp=9        | delta=-2.0
stat_id=  8 | batting:H      | me=13       | opp=23       | delta=-10.0
stat_id= 12 | batting:HR     | me=2        | opp=1        | delta=1.0
stat_id= 13 | batting:RBI    | me=5        | opp=5        | delta=0.0
stat_id= 16 | batting:SB     | me=0        | opp=3        | delta=-3.0
stat_id= 21 | batting:K      | me=24       | opp=15       | delta=9.0
stat_id= 26 | pitching:ERA   | me=3.24     | opp=2.89     | delta=0.35
stat_id= 27 | pitching:WHIP  | me=0.96     | opp=0.86     | delta=0.1
stat_id= 32 | pitching:SV    | me=0        | opp=1        | delta=-1.0
stat_id= 42 | pitching:K     | me=24       | opp=22       | delta=2.0
stat_id= 55 | batting:OPS    | me=.517     | opp=.805     | delta=-0.288
stat_id= 56 | pitching:K/BB  | me=4.80     | opp=5.50     | delta=-0.7
stat_id= 75 | pitching:WIN%  | me=.500     | opp=         | delta=None
s